# 03 Topics: first look

Both methods with a fixed number of topics (`topics.k_initial` in the config), before the k scan:

| | Method 1 | Method 2 |
|---|---|---|
| Vectors | TF-IDF (`text_strong`) | Sentence embeddings (`text_light`) |
| Method | NMF | KMeans |
| Topic words | highest weights in H | c-TF-IDF of the texts in each cluster |
| Assignment | highest weight in W (scaled), below `topics.nmf.min_weight`: no clear topic (-1) | nearest cluster center |

Topics describe what a report is about (the object). How the problem is described is captured separately by state
markers (block E3). That is why state words such as "seit" or "nicht" are not TF-IDF terms (`tfidf.exclude_words`).

Purpose: check whether the topics are meaningful before choosing k.
Requires the preprocessing and vectorization pipelines (see README).

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import pandas as pd

from src.config import load_config
from src.topics.kmeans import central_reports
from src.topics.nmf import NO_TOPIC, top_reports
from src.topics.pipeline import extract_topics
from src.vectorization.pipeline import load_processed

pd.set_option("display.max_colwidth", 200)

cfg = load_config()
K = cfg["topics"]["k_initial"]
STATES = list(cfg["states"])
reports = load_processed(cfg)
result = extract_topics(reports, cfg, K)  # about 1 minute
K

## E1 NMF topics

In [ ]:
def show_table(table):
    """Shares as percentages. duration, repetition, danger: share of the topic's reports with this state."""
    shown = table.copy()
    for column in ["share", *STATES]:
        shown[column] = shown[column].map("{:.1%}".format)
    return shown

show_table(result["nmf_table"])

## E2 KMeans topics

In [ ]:
show_table(result["kmeans_table"])

## E3 State markers per topic

How reports describe the problem, independent of the object (`states` in the config): duration ("seit 3 Wochen"),
repetition ("immer wieder") and danger ("gefährlich"). Share of the reports of each topic with the marker,
darker = larger share.

In [ ]:
overall = result["markers"].mean()
print("share of all reports:", ", ".join(f"{state} {overall[state]:.1%}" for state in STATES))

def state_view(table):
    view = table[STATES].copy()
    first_words = table["words"].str.split(", ").str[:3].str.join(", ")
    view.index = table["topic"].astype(str) + ": " + first_words
    return view.style.format("{:.0%}").background_gradient(cmap="Oranges", axis=None)

state_view(result["nmf_table"])

In [ ]:
state_view(result["kmeans_table"])

## E4 NMF compared with KMeans

Share of the reports of each NMF topic (row) that fall into each KMeans cluster (column).

In [ ]:
crosstab = pd.crosstab(
    pd.Series(result["nmf_labels"], name="NMF topic"),
    pd.Series(result["kmeans_labels"], name="KMeans cluster"),
    normalize="index",
)
crosstab.style.format("{:.0%}").background_gradient(cmap="Blues", axis=None)

## E5 Example reports per topic

Most typical reports: highest NMF weight, or closest to the KMeans cluster center.
Outputs are not committed (nbstripout), because report texts may contain personal data.

In [ ]:
N_EXAMPLES = cfg["topics"]["n_examples"]
nmf_examples = top_reports(result["nmf_weights"], result["nmf_labels"], N_EXAMPLES)
kmeans_examples = central_reports(
    result["embeddings"], result["kmeans_labels"], result["kmeans_model"].cluster_centers_, N_EXAMPLES
)

def show_examples(examples, table):
    for topic in table.loc[table["topic"] != NO_TOPIC, "topic"]:
        words = table.loc[table["topic"] == topic, "words"].iloc[0]
        print(f"Topic {topic}: {words}")
        for i in examples[topic]:
            print("  -", reports.loc[i, "text_light"][:150])
        print()

show_examples(nmf_examples, result["nmf_table"])

In [ ]:
show_examples(kmeans_examples, result["kmeans_table"])

## E6 Number of topics (k scan)

Result of `python -m src.evaluation.k_scan` for k = 5, 10, ..., 40 (see README). Measures:
coherence (do the top words occur together?), diversity (are the topics different?), stability (do fits on
different 80 % samples group the reports the same way?), size of the largest topic and share of reports without a
clear topic (NMF only, KMeans assigns every report).

In [ ]:
from IPython.display import Image

tables_dir = cfg["results"]["tables_dir"]
scan = pd.read_csv(os.path.join("..", tables_dir, "k_scan.csv"))
display(scan.pivot(index="k", columns="method").round(2))
Image(os.path.join("..", cfg["results"]["figures_dir"], "k_scan.png"))

## Observations and decisions

TODO (to be written): Which topics are meaningful? Which topics stand out by duration, repetition or danger?
Which k follows from the k scan, and what does the share of reports without a clear topic mean?